# 05 - Governance network (replaces the router)

Team One Sheep Army. Trains the small network that **automatically decides which modality governs which attribute**.

| | old router (notebook 04) | governance network |
|---|---|---|
| input | 6 global CLIP cosines | frozen CLIP embeddings of caption, object phrase, sketch, reference, grey reference, and the colour part (reference - grey reference) |
| output | 4 global strengths | 3 gates: object conflict, colour stated-and-different, material stated-and-different |
| supervision | soft targets from 9 generated images per item (28 items) | rule labels from the metadata, for thousands of rows; **no image is generated** |

Rules (agreed): geometry and layout from the sketch; object from the text; a colour or material the caption states stays with the text, otherwise the reference supplies it.

Runs in a few minutes; a GPU helps only for the CLIP feature cache.

In [ ]:
# ---- CONFIG + SETUP (same in notebooks 05, 06, 07) ----
DRIVE_ROOT = "/content/drive/MyDrive/sketchy_dataset_one_sketch"
ZIP_PATH = f"{DRIVE_ROOT}/pilot_v1.zip"
LOCAL_DATA = "/content/pilot_v1"
GOV_DIR = f"{DRIVE_ROOT}/governance"            # governance.pt, cached features, fusion LoRA all live here
# The code lives in the repo's src/ folder (cloned below). To use a copy on Drive instead, set REPO_DIR to it.
REPO_URL = "https://github.com/Harshitha-282007/IE643_final.git"
REPO_BRANCH = "governed-pipeline"                # change to "master" once the branch is merged
REPO_DIR = "/content/repo"

import os, sys, glob, json, zipfile
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from google.colab import drive
drive.mount("/content/drive")
if not os.path.isdir(REPO_DIR):
    !git clone -q -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}
assert os.path.exists(f"{REPO_DIR}/src/pipeline.py"), "src/pipeline.py not found: push the new code or point REPO_DIR at it"
sys.path.insert(0, f"{REPO_DIR}/src")
os.makedirs(GOV_DIR, exist_ok=True)
if not glob.glob(f"{LOCAL_DATA}/**/metadata.jsonl", recursive=True):
    os.makedirs(LOCAL_DATA, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(LOCAL_DATA)
DATA_DIR = os.path.dirname(glob.glob(f"{LOCAL_DATA}/**/metadata.jsonl", recursive=True)[0])
BASE_ROWS = [json.loads(l) for l in open(f"{DATA_DIR}/metadata.jsonl") if l.strip()]
print("data:", DATA_DIR, "| base rows:", len(BASE_ROWS))

In [ ]:
!pip install -q -U transformers scikit-learn
import torch, numpy as np, pandas as pd
from tqdm.auto import tqdm
from governance import labels as L, features as FT, model as GM

PER_TYPE = 4          # variants of each kind per photo (object / colour / material / reference swap / agreeing)
rows = L.expand_rows(BASE_ROWS, per_type=PER_TYPE, seed=0)
kind = lambda r: "base" if r["id"] == r["original_id"] else r["id"].split("_")[-1][:3]
df = pd.DataFrame([dict(split=r["split"], kind=kind(r), **r["gates"]) for r in rows])
print("rows:", len(rows), dict(df.split.value_counts()))
print("\nshare of rows with each gate on, by kind (this is what the network has to learn):")
print(df.groupby("kind")[L.GATES].mean().round(2))
print("\nexamples:")
for k in ["base", "obj", "col", "mat", "ref", "agr"]:
    r = next(r for r in rows if kind(r) == k)
    print(f"{k:5s}| {r['caption'][:60]!r} | ref: {r['ref_colors']}, {r['ref_material']} | gates {r['gates']}")

In [ ]:
# ---- cache CLIP ViT-L/14 features (each caption and image is embedded once) ----
FEAT = f"{GOV_DIR}/features_pt{PER_TYPE}.pt"
if os.path.exists(FEAT):
    C = torch.load(FEAT, weights_only=False)
else:
    clip = FT.ClipL()
    C = FT.cache_rows(rows, DATA_DIR, clip, progress=tqdm)
    torch.save(C, FEAT)
    del clip
X, C6, Y = C["X"], C["C6"], C["Y"]
split, group = np.array(C["split"]), np.array(C["group"])
tr, va, te = split == "train", split == "val", split == "test"
print("X", tuple(X.shape), "| train", tr.sum(), "val", va.sum(), "test", te.sum(), "| base photos in test:", len(set(group[te])))

In [ ]:
# ---- 5-fold cross-validation on the train rows, grouped by base photo, against the detector we are replacing ----
dev = "cuda" if torch.cuda.is_available() else "cpu"
Pn, Pc = GM.group_kfold(X[tr], Y[tr], C6[tr], group[tr], k=5, device=dev)
def table(P, Yt, name):
    return pd.DataFrame(GM.scores(P, Yt)).T.round(3).add_prefix(name + " ")
cv = pd.concat([table(Pn, Y[tr], "net"), table(Pc, Y[tr], "6cos")], axis=1)
print("out-of-fold results on train rows (accuracy / F1 / AUROC per gate):")
print(cv[["net acc", "6cos acc", "net f1", "6cos f1", "net auroc", "6cos auroc", "net pos_rate"]])
print("\nReading it: the network should beat the six-cosine baseline clearly on g_col and g_mat. If it does not,")
print("CLIP embeddings do not carry attribute conflicts and the inputs should be switched to Qwen attribute records.")

In [ ]:
# ---- final model on train, checked on val, reported on the held-out test photos ----
net = GM.fit(X[tr], Y[tr], device=dev, Xv=X[va], Yv=Y[va], log=print)
print("calibration (a, b) per gate, fitted on the val photos:", GM.calibrate(net, X[va], Y[va], dev))
Pt = GM.predict(net, X[te], dev)
Pc_t = GM.cosine_baseline(C6[tr], Y[tr], C6[te])
res = pd.concat([table(Pt, Y[te], "net"), table(Pc_t, Y[te], "6cos")], axis=1)
print("\nTEST (held-out photos):")
print(res[["net acc", "6cos acc", "net f1", "6cos f1", "net auroc", "6cos auroc"]])
res.to_csv(f"{GOV_DIR}/E1_detection_test.csv")
GM.save(net, f"{GOV_DIR}/governance.pt", extra=dict(per_type=PER_TYPE, n_train=int(tr.sum())))
print("saved", f"{GOV_DIR}/governance.pt  (copy it to models/governance.pt in the repo)")

In [ ]:
# ---- out-of-vocabulary check: colour / material words replaced by synonyms the label rules do not know ----
# The labels stay those of the original caption. The rule applied to the paraphrase is the word-list detector's score.
para = L.paraphrased([r for r in rows if r["split"] == "test"], seed=0)
PFEAT = f"{GOV_DIR}/features_para_pt{PER_TYPE}.pt"
if os.path.exists(PFEAT):
    Cp = torch.load(PFEAT, weights_only=False)
else:
    clip = FT.ClipL()
    Cp = FT.cache_rows(para, DATA_DIR, clip, progress=tqdm); del clip
    torch.save(Cp, PFEAT)
Pp = GM.predict(net, Cp["X"], dev)
rule = np.array([[L.gates_for(q)[g] for g in L.GATES] for q in para], dtype=float)
oov = pd.concat([table(Pp, Cp["Y"], "net"), table(rule, Cp["Y"], "wordlist")], axis=1)
print(len(para), "paraphrased test rows, e.g.:", para[0]["canonical_caption"], "->", para[0]["caption"])
print(oov[["net acc", "wordlist acc", "net f1", "wordlist f1"]])
oov.to_csv(f"{GOV_DIR}/E1_detection_paraphrased.csv")
print("\nThis is the honest test: the test rows above use the same word lists as the labels, these do not.")
print("(g_obj is not reworded: its wordlist column compares dataset category labels, so it is an oracle, not a baseline.)")